In [ ]:
# # MNPS Job Classification Likelihood Scorer v8.0 – Ultimate Integration
# [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
# 
# **✅ Combines the best of v2 (severity cost), v4 (KSAC-TFIDF + confidence), and v7 (dashboard + workflow)**
# 
# **📁 Expected Inputs:**
# - `Evaluation Resources.zip` containing:
#   - `MNPS KSACs.csv`
#   - `MNPS_Role_Groups_by_KSAC_Similarity_FINAL.csv`
#   - `salary_by_major_role_grouping.csv`
#   - `Time to correct an error in hours.csv`
# - `Sample JDs.csv`
# - `Job_Classifications_Batch.csv`

# ## 1️⃣ Setup Environment and Mount Drive

from google.colab import drive, files
import os
import datetime
import zipfile
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import json
import re
import io
import warnings
warnings.filterwarnings('ignore')

print("🔗 Mounting Google Drive...")
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/MNPS_Likelihood_Analysis_v8'
resources_path = os.path.join(base_path, 'Resources')
results_path = os.path.join(base_path, 'Results')
inputs_path = os.path.join(base_path, 'Inputs')
for path in [base_path, resources_path, results_path, inputs_path]:
    os.makedirs(path, exist_ok=True)

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
run_results_path = os.path.join(results_path, f'Run_v8_{timestamp}')
os.makedirs(run_results_path, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
plt.style.use('seaborn-v0_8-darkgrid')

print(f"✅ Environment ready! Results: {run_results_path}")

# ## 2️⃣ Upload Files

print("📤 Please upload:")
print("1. Evaluation Resources.zip")
print("2. Sample JDs.csv")
print("3. Job_Classifications_Batch.csv")

uploaded = files.upload()

# Process uploaded files
for filename in uploaded.keys():
    print(f"\n📁 Processing: {filename}")
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('/content/temp_extract')
        for root, dirs, files_list in os.walk('/content/temp_extract'):
            for file in files_list:
                src = os.path.join(root, file)
                dst = os.path.join(resources_path, file)
                shutil.copy(src, dst)
        shutil.rmtree('/content/temp_extract', ignore_errors=True)
        print("   ✅ Resources extracted")
    else:
        shutil.copy(filename, os.path.join(inputs_path, filename))
        print("   ✅ Copied to inputs")

# Load data
classifications = pd.read_csv(os.path.join(inputs_path, 'Job_Classifications_Batch.csv'))
job_descriptions = pd.read_csv(os.path.join(inputs_path, 'Sample JDs.csv'))

# Merge on index (assuming alignment by row order)
merged_data = classifications.copy()
if 'Position Summary' in job_descriptions.columns:
    merged_data['Position Summary'] = job_descriptions['Position Summary'].values
if 'job_title_original' not in merged_data.columns and 'new_job_title' in merged_data.columns:
    merged_data['job_title_original'] = merged_data['new_job_title']

print(f"\n✅ Merged {len(merged_data)} records for analysis")

# ## 3️⃣ Load Real Resource Files (from .zip)

# Load KSACs
ksac_file = os.path.join(resources_path, 'MNPS KSACs.csv')
ksac_df = pd.read_csv(ksac_file)
print(f"✅ Loaded KSACs: {len(ksac_df)} roles")

# Build TF-IDF over KSAC_Text
ksac_df['Role'] = ksac_df['Role'].astype(str).str.strip()
ksac_df['KSAC_Text'] = ksac_df['KSAC_Text'].fillna('').astype(str).str.lower()
role_ksac_map = ksac_df.groupby('Role')['KSAC_Text'].apply(' '.join).to_dict()
all_roles = sorted(role_ksac_map.keys())
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
ksac_corpus = [role_ksac_map[r] for r in all_roles]
tfidf_matrix = tfidf.fit_transform(ksac_corpus)
role_to_idx = {role: i for i, role in enumerate(all_roles)}

# Load salary data
salary_file = os.path.join(resources_path, 'salary_by_major_role_grouping.csv')
salary_df = pd.read_csv(salary_file)
salary_df['Major Role Grouping'] = salary_df['Major Role Grouping'].astype(str).str.strip()
global_max_salary = salary_df['Maximum Annual Salary'].max()
global_min_salary = salary_df['Minimum Annual Salary'].min()

# Load time-to-correct
time_file = os.path.join(resources_path, 'Time to correct an error in hours.csv')
time_df = pd.read_csv(time_file)
base_correction_hours = time_df['Average'].iloc[0]  # Assume single value

print("✅ Real resource files loaded")

# ## 4️⃣ v2-Style Severity Cost Functions

def get_ksac_similarity(predicted_role):
    if pd.isna(predicted_role):
        return 0.0
    pred = str(predicted_role).strip()
    if pred not in role_to_idx:
        return 0.0
    pred_vec = tfidf_matrix[role_to_idx[pred]]
    similarities = cosine_similarity(pred_vec, tfidf_matrix).flatten()
    return float(similarities.max())

def get_salary_for_role(role):
    if pd.isna(role):
        return 0.0
    r = str(role).strip()
    match = salary_df[salary_df['Major Role Grouping'] == r]
    if match.empty:
        return (global_min_salary + global_max_salary) / 2.0
    return match['Average Annual Salary'].iloc[0]

def get_normalized_salary(salary):
    if global_max_salary <= global_min_salary:
        return 0.5
    return min(1.0, max(0.0, (salary - global_min_salary) / (global_max_salary - global_min_salary)))

def get_normalized_hours(hours):
    # Normalize by base_correction_hours * 20 (arbitrary high bound)
    return min(1.0, max(0.0, hours / (base_correction_hours * 20)))

def compute_severity_cost_index(ksac_sim, salary, hours):
    dissimilarity = 1.0 - ksac_sim
    salary_norm = get_normalized_salary(salary)
    hours_norm = get_normalized_hours(hours)
    return dissimilarity * (0.7 * salary_norm + 0.3 * hours_norm)

# ## 5️⃣ v4-Style Confidence Engine

def calculate_confidence_components_v4(classification, sub_group, justification, job_description):
    components = {
        'justification_specificity': 0.0,
        'role_indicators': 0.0,
        'subgroup_clarity': 0.0,
        'no_conflicts': 0.0
    }
    # Justification
    if justification and not pd.isna(justification):
        text = str(justification).lower()
        uncertainty = any(w in text for w in ['could', 'might', 'possibly', 'unclear'])
        decisive = sum(w in text for w in ['clearly', 'definitively', 'demonstrates', 'requires'])
        if uncertainty:
            components['justification_specificity'] = 0.1
        elif decisive >= 2:
            components['justification_specificity'] = 0.3
        else:
            components['justification_specificity'] = 0.2

    # Role indicators
    if job_description and not pd.isna(job_description):
        desc = str(job_description).lower()
        keywords = {
            'Manager': ['supervise', 'manage', 'budget', 'lead team'],
            'Coordinator': ['coordinate', 'schedule', 'facilitate'],
            'Teacher': ['teach', 'lesson plan', 'classroom'],
            'Specialist': ['specialized', 'expert', 'consultation'],
            'Technician': ['diagnose', 'troubleshoot', 'repair'],
            'Analyst': ['analyze', 'data', 'report'],
            'Coach': ['coach', 'feedback', 'observe'],
            'Assistant': ['assist', 'support', 'under direction'],
            'Director': ['strategic', 'executive', 'policy']
        }
        if classification in keywords:
            count = sum(kw in desc for kw in keywords[classification])
            components['role_indicators'] = min(0.3, count * 0.06)
        else:
            components['role_indicators'] = 0.15

    # Subgroup clarity
    subgroup_phrases = {'I': ['entry', 'basic'], 'II': ['intermediate'], 'III': ['senior', 'expert']}
    if sub_group in subgroup_phrases and job_description and not pd.isna(job_description):
        desc = str(job_description).lower()
        matches = sum(phrase in desc for phrase in subgroup_phrases[sub_group])
        components['subgroup_clarity'] = min(0.2, 0.1 + matches * 0.05)
    else:
        components['subgroup_clarity'] = 0.1

    # No conflicts
    if job_description and not pd.isna(job_description):
        desc = str(job_description).lower()
        other_roles = [r for r in keywords if r != classification]
        conflict_count = 0
        for r in other_roles:
            matches = sum(kw in desc for kw in keywords[r])
            if matches >= 2:
                conflict_count += 1
        if conflict_count == 0:
            components['no_conflicts'] = 0.2
        elif conflict_count == 1:
            components['no_conflicts'] = 0.1
        else:
            components['no_conflicts'] = 0.05

    total = sum(components.values())
    return total, components

# ## 6️⃣ Process All Records

print("🔄 Processing records with v8.0 engine...")

results = []
for idx, row in merged_data.iterrows():
    classification = row.get('major_role_group', '')
    sub_group = row.get('minor_sub_group', '')
    justification = row.get('grouping_justification', '')
    job_desc = row.get('Position Summary', '')
    job_title = row.get('job_title_original', '')

    # KSAC similarity
    ksac_sim = get_ksac_similarity(classification)

    # Salary & hours
    salary = get_salary_for_role(classification)
    adj_hours = base_correction_hours * (1 + (1 - ksac_sim))  # scales with dissimilarity

    # Severity cost (v2-style)
    severity_cost = compute_severity_cost_index(ksac_sim, salary, adj_hours)

    # Confidence (v4-style)
    confidence, conf_components = calculate_confidence_components_v4(
        classification, sub_group, justification, job_desc
    )

    # Likelihood (simplified v7 mapping)
    quality = ksac_sim * 0.5 + (1 - severity_cost) * 0.5
    if quality >= 0.95:
        likelihood = 4.75 + (quality - 0.95) * 5
    elif quality >= 0.88:
        likelihood = 4.0 + (quality - 0.88) * 10.7
    elif quality >= 0.75:
        likelihood = 3.5 + (quality - 0.75) * 3.85
    elif quality >= 0.5:
        likelihood = 2.0 + (quality - 0.5) * 6.0
    else:
        likelihood = quality * 4.0

    # Performance category
    if likelihood >= 4.75: cat = "Superhuman"
    elif likelihood >= 4.5: cat = "Excellent"
    elif likelihood >= 4.0: cat = "Human-level"
    elif likelihood >= 3.0: cat = "Below Human"
    elif likelihood >= 2.0: cat = "Poor"
    else: cat = "Critical"

    # Review priority
    if likelihood < 3.0 or confidence < 0.3:
        priority = "Critical"
    elif likelihood < 4.0:
        priority = "High"
    elif confidence < 0.6:
        priority = "Medium"
    else:
        priority = "Low"

    results.append({
        'record_id': idx,
        'job_title': job_title,
        'classification': classification,
        'sub_group': sub_group,
        'ksac_similarity': ksac_sim,
        'salary_amount': salary,
        'correction_hours': adj_hours,
        'severity_cost_index': severity_cost,
        'confidence_score': confidence,
        'likelihood_score': likelihood,
        'accuracy_equivalent': quality * 100,
        'performance_category': cat,
        'review_priority': priority,
        **{f'conf_{k}': v for k, v in conf_components.items()}
    })

results_df = pd.DataFrame(results)
print(f"✅ Processed {len(results_df)} records")

# ## 7️⃣ Save v2-Compatible Severity Matrices

# For v2 compatibility, create true vs predicted (assume self-predicted for now)
# In practice, you'd join with ground truth; here we treat classification as prediction
# and use role groups as "true" (for demo)
results_df['true_major_group'] = results_df['classification']
results_df['pred_major_group'] = results_df['classification']

# Confusion matrices
confusion_counts = pd.crosstab(results_df['true_major_group'], results_df['pred_major_group'])
confusion_severity = pd.crosstab(
    results_df['true_major_group'],
    results_df['pred_major_group'],
    values=results_df['severity_cost_index'],
    aggfunc='sum'
).fillna(0.0)

# Save v2 outputs
confusion_severity.to_csv(os.path.join(run_results_path, 'confusion_severity_weighted_v2.csv'))
results_df.to_csv(os.path.join(run_results_path, 'eval_with_severity_index.csv'), index=False)
print("✅ v2-compatible severity matrices saved")

# ## 8️⃣ v7-Style Dashboard

fig = plt.figure(figsize=(22, 16))
gs = gridspec.GridSpec(4, 4, figure=fig, hspace=0.3, wspace=0.3)

# Likelihood distribution
ax = fig.add_subplot(gs[0, 0])
ax.hist(results_df['likelihood_score'], bins=20, alpha=0.7)
ax.set_title('Likelihood Score Distribution')
ax.set_xlabel('Likelihood')
ax.set_ylabel('Frequency')

# Severity cost distribution
ax = fig.add_subplot(gs[0, 1])
ax.hist(results_df['severity_cost_index'], bins=20, alpha=0.7, color='salmon')
ax.set_title('Severity Cost Index Distribution')
ax.set_xlabel('Severity Cost')
ax.set_ylabel('Frequency')

# Performance categories
ax = fig.add_subplot(gs[0, 2])
perf_counts = results_df['performance_category'].value_counts()
ax.barh(perf_counts.index, perf_counts.values, color='lightgreen')
ax.set_title('Performance Categories')

# Review priority
ax = fig.add_subplot(gs[0, 3])
priority_counts = results_df['review_priority'].value_counts()
ax.pie(priority_counts.values, labels=priority_counts.index, autopct='%1.1f%%')
ax.set_title('Review Priority')

# Summary stats
ax = fig.add_subplot(gs[1:, :])
ax.axis('off')
summary = f"""
📊 v8.0 ANALYSIS SUMMARY
Records: {len(results_df)}
Avg Likelihood: {results_df['likelihood_score'].mean():.2f}
Avg Severity Cost: {results_df['severity_cost_index'].mean():.3f}
Total Severity Cost: {results_df['severity_cost_index'].sum():.3f}
Critical Reviews: {(results_df['review_priority'] == 'Critical').sum()}
"""
ax.text(0.1, 0.5, summary, fontsize=12, verticalalignment='center', family='monospace',
        bbox=dict(facecolor='wheat', alpha=0.5))

plt.suptitle(f'MNPS v8.0 Ultimate Integration – {timestamp}', fontsize=16)
viz_path = os.path.join(run_results_path, 'v8_dashboard.png')
plt.savefig(viz_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Dashboard saved: {viz_path}")

# ## 9️⃣ Save Final Results

results_df.to_csv(os.path.join(run_results_path, 'MNPS_Likelihood_Scorer_v8_Results.csv'), index=False)
print(f"\n🎉 v8.0 analysis complete! All results in: {run_results_path}")